# R/S benchmark — PCE training dataset

Builds the training dataset for the classic resistance-load benchmark, using the same machinery as
the carbonation durability pipeline. Its purpose is to have a problem with a **known, cheap** state
limit function, so the emulator chain can be checked against something verifiable and its speed-up
measured.

**State limit function**

$$g = k(t) \cdot \frac{R}{z_1} - S \cdot z_2$$

| symbol | meaning | distribution |
|---|---|---|
| $R$ | resistance | design variable, sampled over its range |
| $S$ | load | design variable, sampled over its range |
| $z_1$ | resistance latent multiplier | normal, mean 1.0, sd 0.028 |
| $z_2$ | load latent multiplier | normal, mean 1.0, sd 0.096 |
| $k(t)$ | degradation factor | $1 + (k_{final} - 1)\,t/100$ |

Failure is $g < 0$. At $t = 0$ we get $k = 1$ and the plain $g = R/z_1 - S z_2$; setting
`k_factor_final = 1.0` removes the time effect altogether.

**Chain of models.** For each design point $(R, S)$ and time step $t$:

1. Draw `n_latent_samples` replicas of $z_1$ and $z_2$ and evaluate $g$ on each — this is the
   expensive Monte Carlo step the surrogate is meant to replace.
2. Fit a **Generalized Lambda Distribution** (GLD, via `pyglam`) to that sample of $g$, compressing
   it into $\lambda_1 \dots \lambda_4$.
3. Fit a **PCE** mapping $(R, S) \mapsto (\lambda_1 \dots \lambda_4)$.

Steps 1-2 live in `emulator_function_time_benchmark`; step 3 and the validation live in
`train_and_validate_pce_at_time_benchmark`. Both are in
[`functions_final.py`](functions_final.py), and mirror their `_durability` counterparts.

**Artefacts written per time step**, following the
`<n_latent_samples>_<kind>_<t>_benchmark.pkl` convention:

| kind | content |
|---|---|
| `dataset_full` | one row per latent replica: $R$, $S$, $z_1$, $z_2$, $g$, lambdas, processing time |
| `dataset_unique` | one row per design point: $R$, $S$, the four lambdas and the processing time |
| `pce_metamodel` | the fitted `PolynomialChaosExpansion` |
| `pce_validation_stats` | MSE and R² per lambda on an independent sample |

**Processing time.** Every design point carries the wall time spent producing it
(`Processing time (s)`), so section 5 can compare the cost of building the dataset against the cost
of evaluating the surrogate that replaces it.

# 1. LIBRARIES

In [ ]:
import os
import sys
import time
from pathlib import Path

# functions_final.py sits in this notebook's own directory
sys.path.insert(0, str(Path.cwd()))

import dill
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib as mpl
import seaborn as sns
from sklearn.metrics import mean_squared_error, r2_score

# Import functions from functions_final.py
from functions_final import *
from UQpy.distributions import Normal, JointIndependent

# Set matplotlib parameters for better aesthetics
mpl.rcParams.update({
                        'font.family': 'serif',
                        'mathtext.fontset': 'cm',
                        'axes.unicode_minus': False,
                        'figure.dpi': 100
                    })

# 2. DEFINITION OF RANDOM VARIABLES

Defines the probability distributions for the input variables:
- Resistance ($R$);
- Load ($S$).

These are the *design* variables the PCE is built over. The latent multipliers $z_1$ and $z_2$ are
not design variables — they are drawn inside the emulator and integrated out by the GLD fit.

### 2.1 Design variables

Normal marginals, matching the benchmark used elsewhere in the repository.

In [ ]:
r_mean  = 5.0   # resistance mean
r_std   = 0.8   # resistance standard deviation
s_mean  = 2.0   # load mean
s_std   = 0.6   # load standard deviation

### 2.2 Fixed parameters for the analysis

`n_latent_samples` is the inner Monte Carlo size used to build the empirical distribution of $g$ at
each design point — it drives the cost of the whole run, and also names the output files.

`k_factor_final` is the degradation factor at $t = 100$. Set it to `1.0` to switch the time effect
off and keep $g = R/z_1 - S z_2$ at every time step.

In [ ]:
n_samples            = 1000     # Number of design samples. Use 1 for testing one sample
n_latent_samples     = 100000   # Number of latent samples per design sample
n_samples_validation = 250      # Number of validation samples
n_lambdas            = 4        # Number of λs to be predicted (λ1, λ2, λ3, λ4)
k_factor_final       = 0.3      # Degradation factor at t = 100. Use 1.0 for no time effect
z1_std               = 0.028    # Standard deviation of the resistance latent multiplier
z2_std               = 0.096    # Standard deviation of the load latent multiplier

### 2.3 Samples

In [ ]:
# Distributions of random variables
r_dist = Normal(loc=r_mean, scale=r_std)
s_dist = Normal(loc=s_mean, scale=s_std)

# Joint distribution
joint = JointIndependent(marginals=[r_dist, s_dist])

# Generate samples
x_pce_rvs = joint.rvs(n_samples)

# Report sample statistics
print("Samples generated successfully!")
print(f"   Number of design samples: {n_samples}")
print(f"   Number of latent samples per design sample: {n_latent_samples}")
print(f"   Total simulations: {n_samples * n_latent_samples}")
print("\nSample statistics:")
print(f"   R:  {x_pce_rvs[:, 0].min():.2f} - {x_pce_rvs[:, 0].max():.2f} (mean: {x_pce_rvs[:, 0].mean():.2f})")
print(f"   S:  {x_pce_rvs[:, 1].min():.2f} - {x_pce_rvs[:, 1].max():.2f} (mean: {x_pce_rvs[:, 1].mean():.2f})")

# 3. SMOKE TEST

One design point at the mean of $R$ and $S$, to confirm the state limit function behaves before
launching the full campaign.

### 3.1 Single design point

In [ ]:
x_single = np.array([[r_mean, s_mean]])

df_single, df_single_unique = emulator_function_time_benchmark(
                                                                  x=x_single,
                                                                  names_x_variables=["r", "s"],
                                                                  time_step=0.0,
                                                                  n_latent_samples=10000,
                                                                  k_factor_final=k_factor_final,
                                                                  z1_std=z1_std,
                                                                  z2_std=z2_std,
                                                              )
df_single.head()

### 3.2 Distribution of the state limit function

At $t = 0$ the degradation factor is 1, so this is $g = R/z_1 - S z_2$ around the nominal point.
`P(g < 0)` is the probability of failure the GLD is being asked to capture.

In [ ]:
print(f"Mean g   = {df_single['g'].mean():.4f}")
print(f"Std  g   = {df_single['g'].std():.4f}")
print(f"P(g < 0) = {np.mean(df_single['g'] < 0):.6f}")
print(f"\nFitted lambdas: {df_single_unique[['lambda 1', 'lambda 2', 'lambda 3', 'lambda 4']].to_numpy()[0]}")

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.hist(df_single['g'], bins=80, color='0.75', edgecolor='none')
ax.axvline(0.0, color='firebrick', lw=1.2, label='$g = 0$ (failure)')
ax.set_xlabel('$g$')
ax.set_ylabel('count')
ax.legend()
plt.tight_layout()
plt.show()

# 4. EVALUATION OF THE BENCHMARK USING GENERALIZED LAMBDA DISTRIBUTION

### 4.1 Time grid

One independent emulator + PCE is built per time step.

In [ ]:
times = np.linspace(0, 150, 10, endpoint=True)  # Time points for the degradation factor
# times = [10, 20, 30, 40]
times

### 4.2 Loop over time steps

`train_and_validate_pce_at_time_benchmark` runs the three stages of one time step and writes the
four artefacts described at the top:

1. evaluates the emulator on `x_pce_rvs` to get the lambdas;
2. fits a PCE of total degree `max_degree` mapping $(R, S)$ to lambdas;
3. re-runs the emulator on a fresh sample of `n_samples_validation` points and scores the PCE with
   MSE and R² per lambda.

Cost warning: each time step runs the emulator twice, over
`(n_samples + n_samples_validation) * n_latent_samples` latent replicas.

In [ ]:
print("="*60)
print("BUILDING THE BENCHMARK EMULATOR")
print("="*60)

results = []
for t in times:
    result = train_and_validate_pce_at_time_benchmark(
                                                         x_train=x_pce_rvs,
                                                         joint=joint,
                                                         time_step=t,
                                                         n_latent_samples=n_latent_samples,
                                                         n_samples_validation=n_samples_validation,
                                                         n_lambdas=n_lambdas,
                                                         k_factor_final=k_factor_final,
                                                         z1_std=z1_std,
                                                         z2_std=z2_std,
                                                         output_dir='.',
                                                     )
    results.append(result)

### 4.3 Validation summary

How well the PCE reproduces each lambda, per time step. Low R² on $\lambda_3$ / $\lambda_4$ (the
tail-shape parameters) is the usual failure mode and is worth checking before trusting the
surrogate downstream.

In [ ]:
validation_summary = pd.concat([r['statistics'] for r in results], ignore_index=True)
validation_summary.insert(0, 'Time (years)', [r['time_step'] for r in results])
validation_summary

# 5. EMULATOR EFFICIENCY (SPEED-UP)

Every design point in `dataset_full` / `dataset_unique` carries a `Processing time (s)` column: the
wall time spent drawing its latent replicas, evaluating $g$ and fitting the GLD.

The comparison below is **cost to build the dataset** versus **cost to evaluate the surrogate that
replaces it**, over the same `n_samples` design points:

- *Emulator (s)* — sum of `Processing time (s)` over all design points at that time step;
- *Surrogate (s)* — wall time of one `pce_metamodel.predict(x_pce_rvs)` call;
- *Speed-up* — the ratio.

This is the payoff of the whole construction: the surrogate is only worth having if that ratio is
large, and it has to be paid for once by the emulator run in 4.2.

### 5.1 Per-design-point cost

In [ ]:
timing_rows = []
for result in results:
    per_point = result['df_unique']['Processing time (s)']
    timing_rows.append({
                           'Time (years)':   result['time_step'],
                           'Total (s)':      per_point.sum(),
                           'Mean (ms)':      per_point.mean() * 1e3,
                           'Min (ms)':       per_point.min() * 1e3,
                           'Max (ms)':       per_point.max() * 1e3,
                       })

emulator_timing = pd.DataFrame(timing_rows)
emulator_timing

### 5.2 Emulator versus surrogate

In [ ]:
speedup_rows = []
for result in results:
    emulator_s = result['emulator_time_s']

    # Cost of the surrogate: one PCE evaluation over the same design points
    t_start = time.perf_counter()
    result['pce_metamodel'].predict(x_pce_rvs)
    surrogate_s = time.perf_counter() - t_start

    speedup_rows.append({
                            'Time (years)':  result['time_step'],
                            'Emulator (s)':  emulator_s,
                            'Surrogate (s)': surrogate_s,
                            'Speed-up':      emulator_s / surrogate_s,
                        })

speedup = pd.DataFrame(speedup_rows)
print(f"Total emulator time over the whole grid: {speedup['Emulator (s)'].sum():.1f} s")
print(f"Median speed-up: {speedup['Speed-up'].median():,.0f}x")
speedup

### 5.3 Speed-up over time

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(speedup['Time (years)'], speedup['Speed-up'], marker='o', color='0.25')
ax.set_xlabel('Time (years)')
ax.set_ylabel('Speed-up (emulator / surrogate)')
ax.set_yscale('log')
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda v, _: f'{v:,.0f}x'))
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()

# 6. NEXT STEPS (work in progress)

Same target as the durability pipeline: collapse the per-time-step PCEs into a single time-aware
model, with the schema:

| inputs | outputs |
|---|---|
| $t$, $R$, $S$ | $\lambda_1$, $\lambda_2$, $\lambda_3$, $\lambda_4$ |

Because the benchmark has a cheap closed-form $g$, this is also where the surrogate can be checked
against brute-force Monte Carlo at points it was never trained on.

In [ ]:
df = []
n_samples_new_dataset = 1000
x_pce_rvs_new_dataset = joint.rvs(n_samples_new_dataset)

for t in times:
    filename = f'{n_latent_samples}_pce_metamodel_{t}_benchmark.pkl'
    with open(filename, 'rb') as f:
        pce_metamodel_new = dill.load(f)
    y_pce_new_dataset_pred = pce_metamodel_new.predict(x_pce_rvs_new_dataset)
    # TODO: stack x_pce_rvs_new_dataset, t and y_pce_new_dataset_pred into a single frame
# TODO: turn the stacked frames into one dataset
# TODO: train a neural network mapping (t, r, s) -> lambdas